In [ ]:
import sys

sys.path.append("..")  # Allow imports from the project root

import torch
from keybert import KeyBERT
from keyphrase_vectorizers import KeyphraseCountVectorizer
from transformers import pipeline

from src.search import ArxivSearchEngine

# Retrieval backend (FAISS + SentenceTransformer live inside this class)

searcher = ArxivSearchEngine(data_dir="../data")

# Local summarization model (used inside tools, NOT as the agent LLM)
setdevice = 0 if torch.cuda.is_available() else -1
summarizer = pipeline(
    "summarization",
    model="sshleifer/distilbart-cnn-12-6",
    device=setdevice,
)

# Keyword extraction (exact Search_Engine.ipynb setup)
# Reuse searcher's embedding model so keywords live in the same vector space
vectorizer = KeyphraseCountVectorizer()
kw_model = KeyBERT(model=searcher.model)

print("Setup complete: retrieval, summarization, and keyword models are ready.")

Initializing Search Engine...
Loading Sentence Transformer model...
Loading FAISS index...
Search Engine Ready!
Setup complete: retrieval, summarization, and keyword models are ready.


## Pipeline Overview

This notebook implements a **LangChain Tool-Calling Agent** over the ArXiv semantic search engine.

```
User Query
    ↓
Groq LLM (orchestrator)
    ↓
Tool selection
    ↓
search_and_summarize  OR  extract_keywords  OR  both
    ↓
LangChain agent loop (automatic ToolMessages)
    ↓
Final human-readable response
```

Retrieval always goes through `src/search.py` (`ArxivSearchEngine.search`).
Summarization and keyword extraction run locally inside the tools.

In [2]:
from langchain_core.tools import tool

#### **1. The Objective**

To convert your raw Hugging Face AI model (the `summarizer` variable you made earlier) into a standardized **LangChain LLM Object**.

#### **2. Why are we doing this? (The "Adapter Plug" Concept)**

Think of **LangChain** as a giant box of Lego blocks for building AI applications. It has blocks for reading PDFs, blocks for searching Vector Databases (like your FAISS index), and blocks for Prompt Templates.

In order to snap all these Lego blocks together into a "Chain", LangChain requires the AI model (the "brain") to have a very specific shape.

* Your old `summarizer` variable was a **Hugging Face** object. If you tried to snap it directly into a LangChain prompt, LangChain would throw an error because they speak different coding languages.
* `HuggingFacePipeline(...)` acts like an **adapter plug** (like using a travel adapter when you visit another country). It takes your Hugging Face model and wraps it in a LangChain shell.



#### **Architecture Split**

This agent uses **three separate models**, each with a clear role:

| Component | Model | Role |
|-----------|-------|------|
| **Agent LLM** | Groq `llama-3.1-8b-instant` | Reasoning, tool selection, final answer |
| **Summarizer** | DistilBART (`sshleifer/distilbart-cnn-12-6`) | Summarize retrieved abstracts inside `search_and_summarize` |
| **Keyword extractor** | KeyBERT + `KeyphraseCountVectorizer` | Extract keyphrases inside `extract_keywords` |

DistilBART and KeyBERT are **tool backends**. They are never wrapped as the agent LLM.

*(Removed: redundant HuggingFacePipeline LLM wrapper. The agent LLM is ChatGroq — see next cell.)*

#### **3. The Result**

By creating the new `llm` variable, you now have an AI model that seamlessly integrates into the LangChain universe.

Instead of just doing basic summarization, you can now use this `llm` variable to:

1. Create a `PromptTemplate` that automatically injects your FAISS search results.
2. Build an `LLMChain` or a RetrievalQA Chain.
3. Hook it directly to an AI Agent.

This single line of code bridges the gap between your local Hugging Face models and advanced RAG architectures without having to pay for OpenAI/ChatGPT API keys!

In [3]:
# Quick smoke test: local DistilBART summarizer (not the agent LLM)
sample = "Deep learning models are widely used for medical image analysis including MRI scan segmentation."
test_summary = summarizer(sample, max_length=40, min_length=10, do_sample=False)
print(test_summary[0]["summary_text"])

Your max_length is set to 40, but your input_length is only 18. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=9)


 Deep learning models are widely used for medical image analysis including MRI scan segmentation .


In [ ]:
import os

from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

# Agent LLM — handles reasoning, tool calling, and final response
# Low temperature keeps tool selection reliable
llm = ChatGroq(
    api_key=os.getenv("GROQ_API_KEY"),
    model="llama-3.1-8b-instant",
    temperature=0,
    max_tokens=1000,
    timeout=30,
    max_retries=6,
)

# Sanity check: verify the Groq connection works
print(llm.invoke("Who won the Cricket World Cup 2011?").content)

India, along with co-hosts Sri Lanka and Bangladesh, hosted the 2011 ICC Cricket World Cup. India won the tournament, defeating Sri Lanka in the final by 6 wickets on April 2, 2011, at the Wankhede Stadium in Mumbai.


**Note:** Tool calling only works after tools are bound via `create_agent` (see below).
Calling `llm.invoke()` directly without tools will return an empty `tool_calls` list — that is expected.

Explanation
load_dotenv() reads the .env file.
os.getenv() retrieves the value of the environment variable.
This keeps secrets out of your source code.

What each parameter means
temperature=0.7

Controls randomness:

0 → Deterministic, consistent outputs.
0.7 → Balanced.
1.0+ → More creative.

max_tokens=1000

Limits the maximum number of output tokens.

Longer responses consume more tokens.

timeout=30

Wait up to 30 seconds for the API before timing out.

max_retries=6

If the request fails due to a temporary network or server issue, LangChain will retry up to 6 times before giving up.

https://docs.langchain.com/oss/python/langchain/models

In [5]:
# from langchain_core.tools import tool

# @tool
# def search_and_summarize(query: str, k: int = 5) -> str:
#     """
#     Search research papers from the FAISS database,
#     retrieve the top-k most similar papers,
#     summarize each paper using BART,
#     and return the results.
#     """

#     # Convert the query into an embedding
#     query_embedding = model.encode([query])

#     # Normalize the embedding for cosine similarity search
#     faiss.normalize_L2(query_embedding)

#     # Search the FAISS index
#     D, I = index.search(query_embedding, k)

#     result = ""

#     # Iterate through the retrieved papers
#     for rank, (score, idx) in enumerate(zip(D[0], I[0]), start=1):

#         # Retrieve the corresponding paper
#         paper = df.iloc[idx]

#         # Generate a summary of the abstract using BART
#         summary = summarizer(
#             paper["abstract"],
#             max_length=120,
#             min_length=40,
#             do_sample=False
#         )[0]["summary_text"]

#         # Format the output
#         result += f"Rank: {rank}\n"
#         result += f"Similarity Score: {round(float(score), 4)}\n"
#         result += f"Title: {paper['title']}\n\n"

#         result += "Abstract:\n"
#         result += paper["abstract"] + "\n\n"

#         result += "Summary:\n"
#         result += summary + "\n\n"

#         result += "-" * 80 + "\n\n"

#     return result

In [6]:
@tool
def search_and_summarize(query: str, k: int = 5) -> str:
    """
    Search the ArXiv FAISS database for papers semantically similar to the query,
    then summarize each retrieved abstract using DistilBART.

    Use this tool when the user asks to find, summarize, or explain research papers.

    Args:
        query: Natural-language research question or topic.
        k: Number of top papers to retrieve (default 5).
    """
    # Step 1: Semantic retrieval via src/search.py
    papers = searcher.search(query, k)

    if not papers:
        return "No papers found for this query."

    # Step 2: Collect abstracts and compute dynamic min_length (Search_Engine.ipynb pattern)
    allabstracts = []
    dynamic_max = 80
    dynamic_min = 20

    for paper in papers:
        abstract_text = paper["abstract"]
        input_word_count = len(abstract_text.split())
        dynamic_min = min(dynamic_min, int(input_word_count))
        allabstracts.append(abstract_text)

    # Step 3: Batch summarization — faster than looping one abstract at a time
    allsummaries = summarizer(
        allabstracts,
        max_length=dynamic_max,
        min_length=dynamic_min,
        batch_size=k,
        do_sample=False,
    )

    # Step 4: Format results for the agent LLM
    result = ""
    for rank, (paper, summary) in enumerate(zip(papers, allsummaries), start=1):
        result += f"Rank: {rank}\n"
        result += f"Similarity Score: {paper['score']:.4f}\n"
        result += f"Title: {paper['title']}\n"
        result += f"Summary: {summary['summary_text']}\n"
        result += f"Abstract (preview): {paper['abstract'][:300]}...\n"
        result += "-" * 80 + "\n\n"

    return result

In [7]:
@tool
def extract_keywords(query: str, k: int = 5) -> str:
    """
    Search the ArXiv FAISS database and extract keyphrases from the top-k papers
    using KeyBERT with KeyphraseCountVectorizer and MMR diversification.

    Use this tool when the user asks for keywords, key phrases, topics, or themes
    related to research papers.

    Args:
        query: Natural-language research question or topic.
        k: Number of top papers to retrieve (default 5).
    """
    # Step 1: Semantic retrieval via src/search.py
    papers = searcher.search(query, k)

    if not papers:
        return "No papers found for this query."

    allabstracts = [paper["abstract"] for paper in papers]

    # Step 2: Keyphrase extraction — exact Search_Engine.ipynb implementation
    keywords = kw_model.extract_keywords(
        allabstracts,
        vectorizer=vectorizer,
        top_n=10,
        use_mmr=True,
        diversity=0.5,  # 0 = pure relevance, 1 = max diversity
    )

    # Step 3: Format results for the agent LLM
    result = ""
    for rank, (paper, paper_keywords) in enumerate(zip(papers, keywords), start=1):
        result += f"Rank: {rank}\n"
        result += f"Similarity Score: {paper['score']:.4f}\n"
        result += f"Title: {paper['title']}\n"
        result += "Keywords:\n"
        for phrase, score in paper_keywords:
            result += f"  - {phrase} ({score:.4f})\n"
        result += "-" * 80 + "\n\n"

    return result

In [ ]:
# Isolated tool smoke tests (not the full agent loop)
test_query = "Explain Retrieval Augmented Generation"

print(" search_and_summarize ")
print(search_and_summarize.invoke({"query": test_query, "k": 3}))

print(" extract_keywords ")
print(extract_keywords.invoke({"query": test_query, "k": 3}))

=== search_and_summarize ===
Rank: 1
Similarity Score: 0.5160
Title: Coordinate-wise Armijo's condition
Summary:  We show by example the advantage of using coordinate-wise Armijo's condition over the usual condition . We then extend results in our recent previous results, on Backtracking-Gradient Descent and some variants, to this setting .
Abstract (preview):   Let $z=(x,y)$ be coordinates for the product space $\mathbb{R}^{m_1}\times
\mathbb{R}^{m_2}$. Let $f:\mathbb{R}^{m_1}\times \mathbb{R}^{m_2}\rightarrow
\mathbb{R}$ be a $C^1$ function, and $\nabla f=(\partial _xf,\partial _yf)$ its
gradient. Fix $0<\alpha <1$. For a point $(x,y) \in \mathbb{R}^{m_...
--------------------------------------------------------------------------------

Rank: 2
Similarity Score: 0.5155
Title: Residual Matrix Product State for Machine Learning
Summary:  Tensor network, which originates from quantum physics, is emerging as an efficient tool for classical and quantum machine learning . ResMPS outperform

In [9]:
from langchain.agents import create_agent

# Both tools are bound to the agent — the LLM decides which to call
tools = [search_and_summarize, extract_keywords]

SYSTEM_PROMPT = """
You are an AI research assistant for Machine Learning ArXiv papers.

You have access to these tools:
1. search_and_summarize — find and summarize relevant research papers.
2. extract_keywords — find papers and extract their keyphrases/topics.

Rules:
- Use search_and_summarize when the user asks to find, summarize, or explain papers.
- Use extract_keywords when the user asks for keywords, key phrases, topics, or themes.
- Use both tools when the user wants a comprehensive analysis of a topic.
- Always ground your final answer in the tool output. Cite paper titles.
- Never invent papers or citations that are not in the tool results.
- Write a clear, concise final response for the user.
"""

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)

print("Agent created with tools:", [t.name for t in tools])

Agent created with tools: ['search_and_summarize', 'extract_keywords']


When LangChain creates an agent, it expects the model to support tool calling.

In [ ]:
# Full agent pipeline: User Query → LLM → Tool(s) → LLM → Final Response
# LangChain handles ToolMessages automatically — no manual construction needed
user_query = "Find the top 3 research papers on Vision Transformer and summarize them."

response = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": user_query}
        ]
    }
)

# The last message is the agent's final grounded answer
final_message = response["messages"][-1]
print("FINAL RESPONSE")
print(final_message.content)

FINAL RESPONSE
Based on the search results, the top 3 research papers on Vision Transformer are:

1. "Direct Runge-Kutta Discretization Achieves Acceleration" by [Author Name]. This paper proposes a novel discretization method for accelerated gradient methods, achieving faster convergence rates.
2. "EEG-TCNet: An Accurate Temporal Convolutional Network for Embedded Motor-Imagery Brain-Machine Interfaces" by [Author Name]. This paper introduces a novel temporal convolutional network (TCN) for brain-machine interfaces, achieving high classification accuracy and low memory requirements.
3. "A Fast and Robust TSVM for Pattern Classification" by [Author Name]. This paper proposes a fast and robust twin support vector machine (TSVM) algorithm for pattern classification, achieving efficient and robust classification performance.

The keywords extracted from these papers include:

* Accelerated gradient method
* Kutta integrators
* Discretization
* Order ODE converges
* Flatness condition
* Li

In [ ]:
# Agent trace (debug): shows the full tool-calling conversation
# HumanMessage → AIMessage (tool_calls) → ToolMessage(s) → AIMessage (final)
for message in response["messages"]:
    print(f"[{type(message).__name__}]")
    if hasattr(message, "tool_calls") and message.tool_calls:
        print(f"  Tool calls: {[tc['name'] for tc in message.tool_calls]}")
    content = message.content
    if content and len(content) > 500:
        print(f"  {content[:500]}...")
    else:
        print(f"  {content}")
    print()

[HumanMessage]
  Find the top 3 research papers on Vision Transformer and summarize them.

[AIMessage]
  Tool calls: ['search_and_summarize']
  

[ToolMessage]
  Rank: 1
Similarity Score: 0.4839
Title: Direct Runge-Kutta Discretization Achieves Acceleration
Summary:  We study gradient-based optimization methods obtained by directly discretizing a second-order ordinary differential equation related to the continuous limit of Nesterov's accelerated gradient method . We show that acceleration can be achieved by a stablelydiscretization of this ODE using standard Runge-Kutta integrators .
Abstract (preview):   We study gradient-based optimization methods obt...

[AIMessage]
  Tool calls: ['extract_keywords']
  

[ToolMessage]
  Rank: 1
Similarity Score: 0.4839
Title: Direct Runge-Kutta Discretization Achieves Acceleration
Keywords:
  - accelerated gradient method (0.5973)
  - kutta integrators (0.4955)
  - discretization (0.3801)
  - order ode converges (0.3023)
  - \mathcal{o}({n^{-2\frac

In [ ]:
# Second query: demonstrates keyword tool selection by the agent
keyword_query = "What are the main keywords and topics in deep learning for medical imaging?"

keyword_response = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": keyword_query}
        ]
    }
)

print("FINAL RESPONSE (keyword query)")
print(keyword_response["messages"][-1].content)

FINAL RESPONSE (keyword query)
Based on the extracted keywords and topics, the main keywords and topics in deep learning for medical imaging are:

1. **Implicit probabilistic models**: This topic is related to the use of implicit models in deep learning for medical imaging, which can be used to model complex distributions and relationships in medical data.
2. **Deep learning for medical imaging**: This topic is a broad area of research that involves the application of deep learning techniques to medical imaging data, such as images and signals.
3. **Neuroimaging data analysis**: This topic is related to the use of deep learning techniques to analyze neuroimaging data, such as functional magnetic resonance imaging (fMRI) data.
4. **Robustness and fairness**: This topic is related to the development of deep learning models that are robust to various types of noise and biases, and that can provide fair and unbiased results.
5. **Attention mechanisms**: This topic is related to the use of 